# Model vs. LLM Gate — Comparison on Unseen Articles (GPU)

Runs the trained nearest-centroid title-gate classifier against **2,103 articles it has never seen** (everything outside the original train+val split — see `backend/export_unseen_titles_for_colab.py`) and compares its decision to what the LLM gate actually decided for each one.

This does on GPU what `backend/compare_model_vs_llm_gate.py` does on CPU inside the backend container — the CPU run was too slow to wait on, so this notebook exists to get the same answer fast.

**Upload two files when prompted**:
1. `backend/unseen_titles_for_colab.csv` — columns `article_id, title, language, llm_gate_decision, published_at`
2. `backend/app/title_gate_centroids.npz` — the two reference vectors (`centroid_relevant`, `centroid_archived`) the production classifier actually uses (see `backend/app/services/title_gate_model.py`)

**What this does and doesn't prove**: `llm_gate_decision` is the LLM gate's own historical output, not independently verified ground truth — same caveat as the original training notebook. This measures agreement between the two mechanisms on data the model was never fit on, not absolute correctness of either one.

In [ ]:
!pip install -q sentence-transformers pandas numpy

In [ ]:
import time
import numpy as np
import pandas as pd
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if device != "cuda":
    print("WARNING: no GPU detected — in Colab, go to Runtime > Change runtime type > GPU.")

In [ ]:
try:
    from google.colab import files
    print("Upload unseen_titles_for_colab.csv...")
    uploaded_csv = files.upload()
    csv_path = next(iter(uploaded_csv))
    print("\nUpload title_gate_centroids.npz...")
    uploaded_npz = files.upload()
    npz_path = next(iter(uploaded_npz))
except ImportError:
    csv_path = "unseen_titles_for_colab.csv"
    npz_path = "title_gate_centroids.npz"
    print(f"Not running in Colab — expecting ./{csv_path} and ./{npz_path}")

In [ ]:
df = pd.read_csv(csv_path)
print(f"Total unseen articles: {len(df)}")
print(df["llm_gate_decision"].value_counts().rename({1: "relevant", 0: "archived"}))

centroids = np.load(npz_path)
centroid_relevant, centroid_archived = centroids["centroid_relevant"], centroids["centroid_archived"]
print(f"\ncentroid_relevant shape: {centroid_relevant.shape}, centroid_archived shape: {centroid_archived.shape}")

## Embed all 2,103 titles on GPU

Same model, same plain `.encode()` call as production (`app/services/title_gate_model.py`) — no instruction prefix, so the comparison stays apples-to-apples.

In [ ]:
from sentence_transformers import SentenceTransformer

EMBEDDING_MODEL_NAME = "Qwen/Qwen3-Embedding-0.6B"
model = SentenceTransformer(EMBEDDING_MODEL_NAME, device=device)

t0 = time.perf_counter()
embeddings = model.encode(df["title"].tolist(), show_progress_bar=True)
elapsed = time.perf_counter() - t0
print(f"\nEmbedded {len(df)} titles in {elapsed:.1f}s ({elapsed/len(df)*1000:.1f}ms/title) on {device}")

## Classify with the exact same nearest-centroid logic production uses

In [ ]:
def normalize(x: np.ndarray) -> np.ndarray:
    norms = np.linalg.norm(x, axis=1, keepdims=True)
    norms[norms == 0] = 1.0
    return x / norms

Xn = normalize(np.asarray(embeddings, dtype=np.float32))
sim_relevant = Xn @ centroid_relevant
sim_archived = Xn @ centroid_archived
df["model_decision"] = (sim_relevant > sim_archived).astype(int)
df["agree"] = (df["model_decision"] == df["llm_gate_decision"]).astype(int)

df.head()

## Results

- **model relevant, LLM archived** — the model is looser than the LLM here; downstream noise if the model's decision had been used instead.
- **model archived, LLM relevant** — the model is stricter than the LLM here; the costlier direction if it happens, since it's the failure mode both the LLM gate and the OR-ensemble were designed to avoid.

In [ ]:
n = len(df)
n_agree = int(df["agree"].sum())
n_model_looser = int(((df["model_decision"] == 1) & (df["llm_gate_decision"] == 0)).sum())
n_model_stricter = int(((df["model_decision"] == 0) & (df["llm_gate_decision"] == 1)).sum())

print(f"Total compared:                                    {n}")
print(f"Agreement:                                         {n_agree}/{n} ({n_agree/n*100:.1f}%)")
print(f"Model relevant, LLM archived (model looser):       {n_model_looser} ({n_model_looser/n*100:.1f}%)")
print(f"Model archived, LLM relevant (model stricter):     {n_model_stricter} ({n_model_stricter/n*100:.1f}%)")

from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score
print(f"\nTreating llm_gate_decision as reference:")
print(f"  accuracy:  {accuracy_score(df['llm_gate_decision'], df['model_decision']):.4f}")
print(f"  precision: {precision_score(df['llm_gate_decision'], df['model_decision']):.4f}")
print(f"  recall:    {recall_score(df['llm_gate_decision'], df['model_decision']):.4f}")
print(f"  f1:        {f1_score(df['llm_gate_decision'], df['model_decision']):.4f}")

## Save the comparison CSV to download

Columns: `article_id, title, language, llm_gate_decision, model_decision, agree, published_at`.

In [ ]:
output_path = "model_vs_llm_gate_comparison.csv"
df.to_csv(output_path, index=False)
print(f"Saved to {output_path}")

try:
    from google.colab import files
    files.download(output_path)
except ImportError:
    print("Not in Colab -- file is saved locally next to this notebook.")